## <B>Kerendia NBA 3.0 HCP Universe</B>
All the HCPs following below conditions qualify for the NBA Universe:
1. <B>HCPs prescribing SGLT2/GLP1 excluding low decile:</B> All HCPs (decile 2 and above) with SGLT2/GLP1 prescriptions in the two years are considered.
2. <B>Kerendia Writers:</B> All HCPs prescribing Kerendia in complete history
3. <B>Nephrologists:</B> All Nephrologists
4. <B>Targeting and BPT HCPs:</B> All HCPs who are part of latest BPT OR Call Plan targeting list have been included.

### <B>Input Sources:</B>
1. PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FCT_RX
2. PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRD
3. PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRVDR
4. PHCDW.PHCDW_TARGETING.CP_KER_HCP
5. PHCDW.PHCDW_DSAA.KER_BPT_2025_Q3

#### <I>Static Sources:</I>
1. <B>Product Group to Name Mapping:</B> /Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_laad_based_lrx/kerendia_laad_based_lrx_volume/Kerendia_Market_Basket_Product_Group_Mapping_and_Priority_v3.xlsx"
2. <B>SGLT2i/GLP1 Mapping:</B> /Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/sglt2_glp1_product_list_df.xlsx

#### <I>Others:</I>
1. Provider ID - NPI - BHID Mapping

### <B>Output</B>:
/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/notebook_output/kerendia_nba_3_hcp_model_base_hcp_universe_for_inference

<I> Documentation Link: [Link](https://confluence.cloud.bayer.com/spaces/USDSCI/pages/717860516/HCP+Universe) </I>

### Loading Libraries and Params

In [0]:
!pip install openpyxl --quiet

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_config"

In [0]:
# Source - 00_Helper_Notebooks/00_config"
data_start = dates["lookback_start"]
data_end = dates["lookback_end"]

### Rx Data

In [0]:
# Loading LAAD Rx Paid Claims
query = f'''
    SELECT DISTINCT A.PATIENT_ID, A.PROVIDER_ID, A.CLAIM_ID, A.SVC_DT, B.PRODUCT_GROUP 
    FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FCT_RX A
    INNER JOIN PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRD B
    USING (NDC_CD)
    WHERE CLAIM_TYPE = 'PD'
    AND SVC_DT >= '{data_start}' 
    AND SVC_DT <= '{data_end}'
'''

paid_claims = Get_Data_Snowflakes(query)
print('Shape :', paid_claims.count(), len(paid_claims.columns))

In [0]:
# Loading LAAD Rx Paid Claims for Kerendia
query = f'''
    SELECT DISTINCT A.PATIENT_ID, A.PROVIDER_ID, A.CLAIM_ID, A.SVC_DT, B.PRODUCT_GROUP 
    FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FCT_RX A
    INNER JOIN PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRD B
    USING (NDC_CD)
    WHERE CLAIM_TYPE = 'PD'
    AND SVC_DT <= '{data_end}'
    AND UPPER(TRIM(PRODUCT_GROUP)) = 'KERENDIA'
'''

all_pats_taking_kerendia = Get_Data_Snowflakes(query)
print('Shape :', all_pats_taking_kerendia.count(), len(all_pats_taking_kerendia.columns))

In [0]:
# Mapping file to map product names to higher level groups
priority_df = pd.read_excel(prod_group_mapping_path)
priority_df = priority_df[['PRODUCT_NAME', 'PRODUCT_GROUP', 'PRIORITY']]
priority_df = spark.createDataFrame(priority_df)
priority_df = priority_df.select("PRODUCT_GROUP","PRODUCT_NAME")

In [0]:
paid_claims = paid_claims.withColumnRenamed("PRODUCT_GROUP","PRODUCT_NAME")
paid_claims = paid_claims.join(priority_df, on='PRODUCT_NAME', how='left')
print('Shape :', paid_claims.count(), len(paid_claims.columns))

In [0]:
paid_claims.select("PRODUCT_NAME","PRODUCT_GROUP").distinct().display()

#### HCPs prescribing Kerendia

In [0]:
# HCPs prescribing Kerendia
all_pats_taking_kerendia = all_pats_taking_kerendia.select("PATIENT_ID", "PROVIDER_ID", "CLAIM_ID")

hcps_writing_kerendia = all_pats_taking_kerendia.select('PROVIDER_ID').distinct()
hcps_writing_kerendia = hcps_writing_kerendia.withColumn("KERENDIA_FLAG",lit(1))
print('Shape :', hcps_writing_kerendia.count(), len(hcps_writing_kerendia.columns))

#### HCPs prescribing SGLT2i/GLP1

In [0]:
# Products belonging to SGLT2/GLP1 class
sglt2_glp1_product_list_df = pd.read_excel(sglt2_glp1_product_list)
# Product Name
sglt2_glp1_product_list_df = spark.createDataFrame(sglt2_glp1_product_list_df)
sglt2_glp1_product_list_df = sglt2_glp1_product_list_df.withColumnRenamed("PRODUCT_GROUP","PRODUCT_NAME")
sglt2_glp1_product_list_df = sglt2_glp1_product_list_df.withColumn("PRODUCT_CLASS",lit("SGLT2_GLP1"))
print("# Products Name :", sglt2_glp1_product_list_df.select("PRODUCT_NAME").distinct().count())

In [0]:
sglt2_glp1_product_list_df.display()

In [0]:
# Filtering for HCPs who prescribed SGLT2/GLP1 products in timeframe of interest
product_list = [row["PRODUCT_NAME"] for row in sglt2_glp1_product_list_df.select("PRODUCT_NAME").collect()]

all_pats_taking_sglt2_glp1 = paid_claims.filter(
    (F.col("SVC_DT") >= data_start) &
    (F.col("SVC_DT") <= data_end) &
    (F.upper(F.trim(F.col('PRODUCT_NAME'))).isin(product_list))
).select("PATIENT_ID","PROVIDER_ID","CLAIM_ID","SVC_DT")

In [0]:
all_pats_taking_sglt2_glp1_df = all_pats_taking_sglt2_glp1.withColumn("MARKET_TRX", lit(1))

# Group by PROVIDER_ID and sum MARKET_TRX
all_pats_taking_sglt2_glp1_df = all_pats_taking_sglt2_glp1_df.groupBy("PROVIDER_ID").agg(
    sum("MARKET_TRX").alias("MARKET_TRX")
)
all_pats_taking_sglt2_glp1_df = all_pats_taking_sglt2_glp1_df.toPandas()

print('Num of HCPs: ', all_pats_taking_sglt2_glp1_df['PROVIDER_ID'].nunique())
print('Shape: ',all_pats_taking_sglt2_glp1_df.shape)
all_pats_taking_sglt2_glp1_df.head()

In [0]:
# Calculate the number of patients per decile
total_mkt_trx = all_pats_taking_sglt2_glp1_df['MARKET_TRX'].sum() 
trx_per_decile = total_mkt_trx / 10

# Sort the DataFrame by Market TRx
all_pats_taking_sglt2_glp1_df = all_pats_taking_sglt2_glp1_df.sort_values(by='MARKET_TRX') 

# Calculate cumulative sum of market TRx
all_pats_taking_sglt2_glp1_df['Cumulative_TRX'] = all_pats_taking_sglt2_glp1_df['MARKET_TRX'].cumsum() 

# Assign decile values to a new column
all_pats_taking_sglt2_glp1_df['Decile'] = (all_pats_taking_sglt2_glp1_df['Cumulative_TRX'] / trx_per_decile).astype(int) + 1
all_pats_taking_sglt2_glp1_df['Decile'].replace(11,10,inplace=True)

print('Shape: ',all_pats_taking_sglt2_glp1_df.shape)
all_pats_taking_sglt2_glp1_df.head()

In [0]:
all_pats_taking_sglt2_glp1_df['Decile'].value_counts()

In [0]:
# all_pats_taking_sglt2_glp1_df[all_pats_taking_sglt2_glp1_df['Decile']==2]['MARKET_TRX'].describe()

In [0]:
all_pats_taking_sglt2_glp1_df = all_pats_taking_sglt2_glp1_df[['PROVIDER_ID','Decile']]
all_pats_taking_sglt2_glp1_df = all_pats_taking_sglt2_glp1_df[all_pats_taking_sglt2_glp1_df['Decile'].isin([2,3,4,5,6,7,8,9,10])]
print('Shape: ',all_pats_taking_sglt2_glp1_df.shape)

In [0]:
hcps_writing_sglt2_glp1 = spark.createDataFrame(all_pats_taking_sglt2_glp1_df)
hcps_writing_sglt2_glp1 = hcps_writing_sglt2_glp1.select('PROVIDER_ID').distinct()
hcps_writing_sglt2_glp1 = hcps_writing_sglt2_glp1.withColumn("SGLT2_GLP1_FLAG",lit(1))
print('Shape :', hcps_writing_sglt2_glp1.count(), len(hcps_writing_sglt2_glp1.columns))

#### Neph HCPs

In [0]:
# Read DIM Provider Table
query = '''
          SELECT DISTINCT PROVIDER_ID
          FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRVDR
        '''
provider_attributes = Get_Data_Snowflakes(query)
print('Shape :', provider_attributes.count(), len(provider_attributes.columns))
print('# Distinct Providers :', provider_attributes.select("PROVIDER_ID").distinct().count())

In [0]:
# BHID-NPI Mapping
query = '''
            SELECT DISTINCT SRCNBRX.PRVDR_ID AS PROVIDER_ID, SRCNBRXIMS.NPI AS NPI_NUMBER,
            CASE
                -- Case 1: Where available, bring BHID from FACT_IQV_LAAD_SUB_NATIONAL_XREF
                WHEN MAPCUSTXREF.BAYER_CUST_ID IS NOT NULL THEN MAPCUSTXREF.BAYER_CUST_ID
                -- Case 2: If BHID not available in above table, bring from DIM_CUSTOMER_HCP
                WHEN ((TRIM(SRCNBRXIMS.NPI) != 'NULL' OR SRCNBRXIMS.NPI IS NOT NULL OR TRIM(SRCNBRXIMS.NPI) != '') 
                AND (DIMCUST.BAYER_CUST_ID IS NOT NULL OR DIMCUST.BAYER_CUST_ID!='')) THEN DIMCUST.BAYER_CUST_ID
                -- Case 3: If Provider_ID is Null, then flow in Provider_ID
                WHEN (TRIM(SRCNBRXIMS.PRVDR_ID) = 'NULL') THEN SRCNBRXIMS.PRVDR_ID
                ELSE COALESCE(SRCNBRXIMS.PRVDR_ID, SRCNBRX.PRVDR_ID)
                END AS BH_ID
            FROM 
            (
                -- LAAD NBRx
                SELECT DISTINCT * FROM CPH_DB_PROD.MODEL_V2.FACT_IQV_LAAD_SUB_NATIONAL 
                WHERE PROJD_NBRX > 0
            ) SRCNBRX
            LEFT JOIN
            -- XREF
            CPH_DB_PROD.MODEL_V2.FACT_IQV_LAAD_SUB_NATIONAL_XREF SRCNBRXIMS
            ON SRCNBRX.PRVDR_ID = SRCNBRXIMS.PRVDR_ID
            LEFT JOIN
            (
                -- Source 1: Extract BH_ID
                SELECT SOURCE_CUST_ID, SOURCE_CUST_ID_TYP, MIN(BAYER_CUST_ID) AS BAYER_CUST_ID 
                FROM CPH_DB_PROD.MODEL_V2.MAP_CUSTOMER_XREF
                WHERE SOURCE_CUST_ID_TYP IN ('IQV_LAAD')
                GROUP BY 1, 2
            ) AS MAPCUSTXREF
            ON SRCNBRXIMS.PRVDR_ID = MAPCUSTXREF.SOURCE_CUST_ID
            LEFT JOIN 
            (
                -- Source 2: Extract BH_ID
                SELECT MIN(BAYER_CUST_ID) AS BAYER_CUST_ID, (NPI_NUM) NPI_NUM 
                FROM CPH_DB_PROD.MODEL_V2.DIM_CUSTOMER_HCP 
                GROUP BY 2
            ) AS DIMCUST
            ON SRCNBRXIMS.NPI = DIMCUST.NPI_NUM
        '''
npi_provider_bhid_mappig = Get_Data_Snowflakes(query)
print("Shape :", npi_provider_bhid_mappig.count(), len(npi_provider_bhid_mappig.columns))

In [0]:
npi_provider_bhid_mappig.write.option("overwriteSchema", "true").mode("overwrite").saveAsTable("`ph-com-ai-us-prod-catalog-2553213839599380`.kerendia_nba_3.SOURCE_PROVIDER_BHID_NPI_MAPPING")

In [0]:
npi_provider_bhid_mappig.write.format("delta").mode("overwrite").save("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_0_npi_bhid_mapping")

In [0]:
# Add BHID to map MDM PRFL table
provider_attributes = provider_attributes.join(npi_provider_bhid_mappig, on="PROVIDER_ID", how='left')
print("Shape :", provider_attributes.count(), len(provider_attributes.columns))
print("# Distint HCPs :", provider_attributes.select("PROVIDER_ID").distinct().count())

In [0]:
# Extract HCP profile data & Specialty from PRFL table
query = '''
          SELECT DISTINCT BH_ID, SPECIALTY_1 AS SPECIALTY_CODE
          FROM PHCDW.PHCDW_STG.MDM_STG_HCP_PRFL
        '''
mdm_hcp_profile = Get_Data_Snowflakes(query)
print('Shape :', mdm_hcp_profile.count(), len(mdm_hcp_profile.columns))

In [0]:
# Add Specialty Code from MDM PRFL table
provider_attributes = provider_attributes.join(mdm_hcp_profile, on="BH_ID", how='left')
print("Shape :", provider_attributes.count(), len(provider_attributes.columns))
print("# Distint HCPs :", provider_attributes.select("PROVIDER_ID").distinct().count())

In [0]:
# Loading SPecialty Code to Specialty Group Mapping - Source Data Team/Galaxy
sp_grp_to_sp_mapping = spark.read.csv(specialty_cd_to_group_mapping_path, inferSchema=True, header=True)
sp_grp_to_sp_mapping = sp_grp_to_sp_mapping.drop("COLUMN (NBRX, TRXS) To be added", "Specialty Desc")
sp_grp_to_sp_mapping = sp_grp_to_sp_mapping.withColumnRenamed("Specialty Grouping (Non ONC)", "SPECIALTY_GROUP").withColumnRenamed("Specialty Code", "SPECIALTY_CODE")
print('Shape :', sp_grp_to_sp_mapping.count(), len(sp_grp_to_sp_mapping.columns))
sp_grp_to_sp_mapping.show(5)

In [0]:
# Mapping Specialty Codes to Specialty Group
provider_attributes = provider_attributes.join(sp_grp_to_sp_mapping, on='SPECIALTY_CODE', how='left')
provider_attributes = provider_attributes.drop("SPECIALTY_CODE", "BH_ID", "NPI_NUMBER")
print('Shape :', provider_attributes.count(), len(provider_attributes.columns))
print("# Distint HCPs :", provider_attributes.select("PROVIDER_ID").distinct().count())

In [0]:
provider_attributes = provider_attributes.select(['PROVIDER_ID','SPECIALTY_GROUP']).distinct()
neph_hcps = provider_attributes.filter(col("SPECIALTY_GROUP") == "NEPH").select("PROVIDER_ID").distinct()
neph_hcps = neph_hcps.withColumn("NEPH_FLAG",lit(1))
neph_hcps.count()

#### Targeting List HCPs

In [0]:
targeting_hcps = Get_Data_Snowflakes('''
                                        SELECT DISTINCT BAYER_ID as BH_ID
                                        FROM PHCDW.PHCDW_TARGETING.CP_KER_HCP
                                        WHERE FF in ('CR1', 'CR2', 'CR4', 'CR5')
                                        AND ((UPPER(PRODUCT_1) = 'KERENDIA') or (UPPER(PRODUCT_2) = 'KERENDIA'))
                                    ''')
targeting_hcps = targeting_hcps.join(npi_provider_bhid_mappig.select(['PROVIDER_ID','BH_ID']).distinct(), on='BH_ID',how='inner')
targeting_hcps = targeting_hcps.select("PROVIDER_ID").distinct()
targeting_hcps = targeting_hcps.withColumn("TARGETING_HCP_FLAG",lit(1))
print('Shape :', targeting_hcps.count(), len(targeting_hcps.columns))

In [0]:
# targeting_hcps = spark.read.csv(CP_KER_HCP_PATH, inferSchema = True, header = True)
# targeting_hcps = targeting_hcps.withColumnRenamed("HCP_ID", "BH_ID")
# targeting_hcps = targeting_hcps.join(npi_provider_bhid_mappig.select(['PROVIDER_ID','BH_ID']).distinct(), on='BH_ID',how='inner')
# targeting_hcps = targeting_hcps.select("PROVIDER_ID").distinct()
# targeting_hcps = targeting_hcps.withColumn("TARGETING_HCP_FLAG",lit(1))
# print('Shape :', targeting_hcps.count(), len(targeting_hcps.columns))

#### BPT HCPs

In [0]:
# bpt_hcps = Get_Data_Snowflakes(f'''SELECT DISTINCT BH_ID FROM {bpt_table_name}''')
# bpt_hcps = spark.read.csv(bpt_flat_file_path, inferSchema=True, header=True)
# Update: CP_KER_HCP is source for BPT List HCPs
query = """
  SELECT DISTINCT BAYER_ID as BH_ID
  FROM PHCDW.PHCDW_TARGETING.CP_KER_HCP
  WHERE FF in ('CR1', 'CR2', 'CR4', 'CR5')
  AND ((UPPER(PRODUCT_1) = 'KERENDIA') or (UPPER(PRODUCT_2) = 'KERENDIA'))
"""

bpt_hcps = Get_Data_Snowflakes(query)
# bpt_hcps = bpt_hcps.withColumnRenamed("HCP_ID","BH_ID").select("BH_ID").distinct()

# BPT HCPs before PROVIDER_ID mapping
bpt_hcps = bpt_hcps.select("BH_ID").distinct()
bpt_hcps_before_mapping = bpt_hcps.count()

# Map BH_ID to PROVIDER_ID
bpt_hcps = bpt_hcps.join(
    npi_provider_bhid_mappig.select(['PROVIDER_ID', 'BH_ID']),
    on='BH_ID',
    how='inner'
)

# BPT HCPs after PROVIDER_ID mapping
bpt_hcps_after_mapping = bpt_hcps.count()

# HCPs that could not be mapped
bpt_hcps_not_mapped = bpt_hcps_before_mapping - bpt_hcps_after_mapping

bpt_hcps = bpt_hcps.withColumn("BPT_HCP_FLAG", lit(1))

print("BPT HCPs before mapping  :", bpt_hcps_before_mapping)
print("BPT HCPs after mapping   :", bpt_hcps_after_mapping)
print("Unmapped BH_IDs (dropped):", bpt_hcps_not_mapped)
print("Shape :", bpt_hcps.count(), len(bpt_hcps.columns))

#### Combining HCP Universe

In [0]:
from functools import reduce

dfs = [
    hcps_writing_kerendia,
    hcps_writing_sglt2_glp1,
    neph_hcps,
    targeting_hcps,
    bpt_hcps
]

# Function for full outer join on PROVIDER_ID
def outer_join(df_a: DataFrame, df_b: DataFrame) -> DataFrame:
    return df_a.join(df_b, on="PROVIDER_ID", how="outer")

df_hcp_universe = reduce(outer_join, dfs)
df_hcp_universe = df_hcp_universe.fillna(0)
df_hcp_universe = df_hcp_universe.drop("BH_ID")

print('Shape :', df_hcp_universe.count(), len(df_hcp_universe.columns))

In [0]:
df_hcp_universe = df_hcp_universe.dropDuplicates(["PROVIDER_ID"])
print('Shape :', df_hcp_universe.count(), len(df_hcp_universe.columns))
df_hcp_universe.display()

In [0]:
agg_exprs = [F.sum(c).alias(f"total_{c}") for c in df_hcp_universe.columns[1:]]
df_hcp_universe.agg(*agg_exprs).display()

In [0]:
# path = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/notebook_output/kerendia_nba_3_hcp_model_base_hcp_universe_for_inference"

# if os.path.exists(path):
#     shutil.rmtree(path)
#     print("Removed!")
# else:
#     print("File not found!")

In [0]:
curr_date = str(dt.today().strftime('%Y_%m_%d'))
print(curr_date)

df_hcp_universe = df_hcp_universe.withColumn("PULL_DT", F.lit(curr_date))

# Overwrite the table
df_hcp_universe.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/dev_folder_3_1/notebook_output/kerendia_nba_3_hcp_model_base_hcp_universe_for_inference")

In [0]:
# # Saving a Backup Copy as well with date identifier
# file_name = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline_snapshots/hcp_universe/kerendia_nba_3_hcp_model_base_hcp_universe_for_inference_"+curr_date+".parquet"
# print("File Name:", file_name)
# df_hcp_universe.write.mode("overwrite").parquet(file_name)
# print('File Saved!')